# Introduction: Encoding Time Series Features

In this notebook, we will explore the options for encoding times and dates in a time series problem. The primary objective is to determine the optimal method for representing time in a time-series problem, particularly as it relates to building energy data.

In [ ]:
# Standard Data Science Helpers
import numpy as np
import pandas as pd
import os

from plotly.offline import iplot, init_notebook_mode
init_notebook_mode(connected=True)

import cufflinks as cf
cf.set_config_file(world_readable=True, theme="pearl")
cf.go_offline(connected=True)

# Extra options
pd.options.display.max_rows = 10
pd.options.display.max_columns = 30
# Show all code cells outputs
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [ ]:
%load_ext autoreload
%autoreload 2

from warnings import filterwarnings
filterwarnings('ignore', category=RuntimeWarning)

In [ ]:
from notebooks import create_time_features

In [ ]:
data = pd.read_csv('data/building_1.csv', parse_dates=['timestamp'], 
                    usecols=['timestamp', 'energy'])
data.head()

In [ ]:
# Remove missing values
data = data.dropna(subset=['energy'])

# Count frequencies
freq_counts = data['timestamp'].diff(1).value_counts()
# Most common frequency in minutes
freq = round(freq_counts.idxmax().total_seconds() / 60)
freq

In [ ]:
data = data.set_index('timestamp').sort_index()

In [ ]:
data = pd.concat(
    [data, create_time_features(data.index, cyc_encode=True)], axis=1)
data.columns

In [ ]:
data[[c for c in data if ('sin' in c) or ('cos' in c)]].describe()

In [ ]:
data.head()

In [ ]:
data.loc[(data.index.week == 1) & (data.index.year == 2015), 'timestamp_fracday'].iplot()

In [ ]:
data.loc[(data.index.month == 1) & (data.index.year == 2015), 'timestamp_fracweek'].iplot()

In [ ]:
data.loc[(data.index.year == 2015), 'timestamp_fracmonth'].iplot()

In [ ]:
data.loc[:, 'timestamp_fracyear'].iplot()

In [ ]:
data.loc[(data.index.week == 1) & (data.index.year == 2015), ['sin_timestamp_fracday', 'cos_timestamp_fracday']].iplot()

In [ ]:
data.loc[(data.index.month == 1) & (data.index.year == 2015), ['sin_timestamp_fracweek', 'cos_timestamp_fracweek']].iplot()

In [ ]:
data.loc[(data.index.year == 2015), ['sin_timestamp_fracmonth', 'cos_timestamp_fracmonth']].iplot()

In [ ]:
data.loc[:, ['sin_timestamp_fracyear', 'cos_timestamp_fracyear']].iplot()

In [ ]:
np.unique(data.index.day)

In [ ]:
def data_reading(building_filename):
    """
    Read in building energy data from csv file
    """
    data = pd.read_csv(building_filename, parse_dates=['timestamp'], 
                    usecols=['timestamp', 'energy'])
    # Remove missing values
    data = data.dropna(subset=['energy'])
    
    # Count frequencies
    freq_counts = data['timestamp'].diff(1).value_counts()
    # Most common frequency in minutes
    freq = round(freq_counts.idxmax().total_seconds() / 60)
    
    # Set the index
    data = data.set_index('timestamp').sort_index()
    return data, freq, len(data)

In [ ]:
def data_testing(building_filename, model, splits=4):
    """
    Test the model on the building energy data in a csv file
    
    :param filename: string filename of building
    :param model: sklearn compatible model
    :param splits: integer number of time series splits (default is 4)
    
    :return: dataframe of results
    """
    # File the building id for indexing
    building_id = building_filename.split('_')[-1].split('.csv')[0]
    # Read in the file
    data, freq, dpoints = data_reading(building_filename)
    # Test the model on the data
    results = test_time_date_features(data, model, splits=splits)
    
    model_name = type(model).__name__
    
    # Record the results
    results['freq'] = freq
    results['dpoints'] = dpoints
    results['building_id'] = building_id
    results['model'] = model_name
    results['splits'] = splits
    return results

In [ ]:
def mape(y_true, y_pred):
    # The mean absolute percentage error between true values and prediction
    return 100 * np.mean(np.abs((y_pred - y_true) / y_true))

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

def test_time_date_features(data, model, splits=4):
    """
    Test different time and date encoding methods on building energy data
    
    
    :param data: building energy dataset to use
    :param model: sklearn compatible model
    :param splits: integer number of splits for time series evaluation (default is 4)
    
    :return: dataframe of results
    
    """

    # Create the time features
    data = pd.concat(
        [data, create_time_features(data.index, cyc_encode=True)], axis=1)

    # Targets
    y = data.pop('energy')

    # Each set of features
    
    # Baseline typical
    baseline_features = [
        'timestamp_' + t
        for t in ['minute', 'hour', 'dayofweek', 'day', 'week', 'month', 'dayofyear', 'year']
    ]
    
    # Cyclical encoding of baseline
    baseline_cyc_features = [
        'sin_' + t for t in baseline_features
        if t not in ['timestamp_dayofyear', 'timestamp_year', 'timestamp_minute']
    ] + [
        'cos_' + t for t in baseline_features
        if t not in ['timestamp_dayofyear', 'timestamp_year', 'timestamp_minute']
    ]

    # Fractional features
    frac_features = [
        'timestamp_' + t
        for t in ['fracday', 'fracweek', 'fracmonth', 'fracyear']
    ]
    
    # Cyclical encoding of fractional
    frac_cyc_features = ['sin_' + t for t in frac_features
                         ] + ['cos_' + t for t in frac_features]

    # Intuition based features
    # Cylical time of day
    # Day of week
    # Cyclical time of year
    domain_features = [
        'sin_timestamp_fracday', 'cos_timestamp_fracday', 'timestamp_dayofweek',
        'sin_timestamp_fracyear', 'cos_timestamp_fracyear'
    ]

    # Dictionary to hold results
    results = {}
    
    # Lists for storing results
    scores = []
    stds = []
    methods = []
    test_points = []
    method_names = ['baseline', 'baseline_cyc', 'frac', 'frac_cyc', 'domain']

    tss = TimeSeriesSplit(n_splits=splits)
    
    # Iterate through each set of features
    for features, name in zip(
        [baseline_features, baseline_cyc_features, frac_features, frac_cyc_features, domain_features],
            method_names):
        
        # Create a new list to hold results for each method
        method_scores = []
        dataset = data[features].copy()
        
        # Drop any columns with only 1 value
        to_drop = dataset.columns[(dataset.nunique() == 1)]
        dataset = dataset.drop(columns=to_drop)
        

        # Time series based evaluation
        for train_idx, test_idx in tss.split(dataset):

            X_train, X_test = dataset.iloc[train_idx], dataset.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            
            mape_value = mape(y_test, predictions)
            # Record score as an accuracy so higher is better
            method_scores.append(100 - mape_value)
        
         # Find the stats for the method
        scores.append(np.median(method_scores))
        stds.append(np.std(method_scores))
        methods.append(name)
        
        # Number of testing points is always the same
        test_points.append(len(test_idx))
    
    results = pd.DataFrame(dict(score=scores, std=stds, method=methods, test_points=test_points))
    return results

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor


linear_model = LinearRegression()
random_model = RandomForestRegressor(random_state=42, n_estimators=10, n_jobs=-1)
rl = data_testing('data/building_1.csv', linear_model, splits=5)
rf = data_testing('data/building_1.csv', random_model, splits=5)
rl.append(rf)

In [ ]:
linear_model = LinearRegression()
random_model = RandomForestRegressor(random_state=42, n_estimators=100, n_jobs=-1)
rl = data_testing('data/building_1.csv', linear_model, splits=5)
rf = data_testing('data/building_1.csv', random_model, splits=5)
rl.append(rf)

In [ ]:
linear_model = LinearRegression()
random_model = RandomForestRegressor(random_state=42, n_estimators=10, max_depth=10, n_jobs=-1)
rl = data_testing('data/building_1.csv', linear_model, splits=5)
rf = data_testing('data/building_1.csv', random_model, splits=5)
rl.append(rf)

In [ ]:
linear_model = LinearRegression()
random_model = RandomForestRegressor(random_state=42, n_estimators=100, max_depth=10, n_jobs=-1)
rl = data_testing('data/building_1.csv', linear_model, splits=5)
rf = data_testing('data/building_1.csv', random_model, splits=5)
rl.append(rf)

In [ ]:
from tqdm import tqdm_notebook

linear_model = LinearRegression()
random_model = RandomForestRegressor(random_state=42, n_estimators=10, max_depth=10, n_jobs=-1)

def run_all_buildings():
    results = []
    for file in tqdm_notebook(os.listdir('data')):
        if file.endswith('.csv'):
            results.append(data_testing(f'data/{file}', linear_model, splits=5))
            results.append(data_testing(f'data/{file}', random_model, splits=5))
            
    results = pd.concat(results)
    results.to_csv(f'results/results.csv')
    return results

In [ ]:
results = run_all_buildings()

In [ ]:
all_linear_results.groupby('building_id').apply(lambda x: x.loc[x['score'].idxmax(), 'method']).value_counts()

In [ ]:
all_linear_results.pivot_table(index='building_id', columns='method', values='score').iplot()

In [ ]:
all_linear_results.set_index('building_id').iplot(y='score', categories='method')

In [ ]:
all_random_results.groupby('building_id').apply(lambda x: x.loc[x['score'].idxmax, 'method']).value_counts()